In [1]:
import pandas as pd
from pathlib import Path

DATA_PROCESSED = Path("../data/processed")

In [2]:
hr = pd.read_csv(DATA_PROCESSED / "hr_cleaned.csv")

risk = pd.read_csv(
    DATA_PROCESSED / "attrition_risk_predictions.csv"
)

print("HR data:", hr.shape)
print("Risk data:", risk.shape)

HR data: (3000, 42)
Risk data: (294, 5)


In [3]:
attrition = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

hr_risk = risk.merge(
    attrition,
    on="EmployeeNumber",
    how="left"
)

print("HR Risk Dataset:", hr_risk.shape)

HR Risk Dataset: (294, 39)


In [4]:
hr_risk["RiskLevel"] = pd.Categorical(
    hr_risk["RiskLevel"],
    categories=["Low", "Medium", "High"],
    ordered=True
)

print(hr_risk["RiskLevel"].value_counts().sort_index())

RiskLevel
Low       238
Medium     41
High       15
Name: count, dtype: int64


In [5]:
high_risk = hr_risk[
    hr_risk["RiskLevel"] == "High"
].copy()

print("High-risk employees:", len(high_risk))

print(
    high_risk[
        [
            "EmployeeNumber",
            "AttritionProbability",
            "JobRole",
            "Department",
            "JobLevel",
            "MonthlyIncome",
            "OverTime",
            "JobSatisfaction",
            "WorkLifeBalance"
        ]
    ].sort_values(
        "AttritionProbability",
        ascending=False
    ).head(15)
)

High-risk employees: 15
     EmployeeNumber  AttritionProbability                    JobRole  \
92              121              0.975228            Sales Executive   
200             274              0.956848     Manufacturing Director   
223             307              0.942481            Sales Executive   
214             297              0.896456         Research Scientist   
276             381              0.813421                    Manager   
199             273              0.812851     Manufacturing Director   
135             178              0.735263     Manufacturing Director   
168             230              0.712330            Sales Executive   
81              105              0.694058         Research Scientist   
158             215              0.647253            Sales Executive   
251             343              0.643718  Healthcare Representative   
204             282              0.560069  Healthcare Representative   
35               46              0.55254

In [6]:
department_risk = (
    hr_risk.groupby("Department", observed=True)
    .agg(
        Employees=("EmployeeNumber", "count"),
        AvgAttritionRisk=("AttritionProbability", "mean"),
        HighRiskEmployees=("RiskLevel", lambda x: (x == "High").sum())
    )
    .sort_values("AvgAttritionRisk", ascending=False)
)

department_risk["AvgAttritionRisk"] *= 100

print(department_risk.round(2))

                        Employees  AvgAttritionRisk  HighRiskEmployees
Department                                                            
Sales                          77             16.73                  4
Research & Development        210             14.15                 11
Human Resources                 7              3.97                  0


In [7]:
overtime_risk = (
    hr_risk.groupby("OverTime", observed=True)
    ["AttritionProbability"]
    .mean()
    .mul(100)
    .round(2)
)

print("Average attrition risk by overtime:")
print(overtime_risk)

Average attrition risk by overtime:
OverTime
No     15.19
Yes    13.12
Name: AttritionProbability, dtype: float64


In [8]:
def recommend_action(row):
    if row["RiskLevel"] == "High":
        if row["OverTime"] == "Yes":
            return "Review workload and overtime"
        elif row["JobSatisfaction"] <= 2:
            return "Conduct employee satisfaction discussion"
        elif row["WorkLifeBalance"] <= 2:
            return "Review work-life balance"
        else:
            return "Schedule HR retention discussion"

    if row["RiskLevel"] == "Medium":
        return "Monitor engagement and satisfaction"

    return "Continue regular engagement"

hr_risk["HRRecommendation"] = hr_risk.apply(
    recommend_action,
    axis=1
)

print(
    hr_risk[
        [
            "EmployeeNumber",
            "AttritionProbability",
            "RiskLevel",
            "HRRecommendation"
        ]
    ].head(10)
)

   EmployeeNumber  AttritionProbability RiskLevel  \
0               1              0.287895    Medium   
1               2              0.022285       Low   
2               4              0.117092       Low   
3               5              0.009519       Low   
4               7              0.283012    Medium   
5               8              0.092380       Low   
6              10              0.071071       Low   
7              11              0.050783       Low   
8              12              0.015427       Low   
9              13              0.434828    Medium   

                      HRRecommendation  
0  Monitor engagement and satisfaction  
1          Continue regular engagement  
2          Continue regular engagement  
3          Continue regular engagement  
4  Monitor engagement and satisfaction  
5          Continue regular engagement  
6          Continue regular engagement  
7          Continue regular engagement  
8          Continue regular engagement  
9  Mon

In [9]:
output = DATA_PROCESSED / "hr_intelligence.csv"

hr_risk.to_csv(output, index=False)

print("Saved:", output)
print("Shape:", hr_risk.shape)

Saved: ..\data\processed\hr_intelligence.csv
Shape: (294, 40)


In [10]:
print("=" * 70)
print("HR INTELLIGENCE COMPLETE")
print("=" * 70)

print("Employees:", len(hr_risk))
print("\nRisk distribution:")
print(hr_risk["RiskLevel"].value_counts())

print("\nHigh-risk employees:", 
      (hr_risk["RiskLevel"] == "High").sum())

print("\nOutput:")
print(output)

HR INTELLIGENCE COMPLETE
Employees: 294

Risk distribution:
RiskLevel
Low       238
Medium     41
High       15
Name: count, dtype: int64

High-risk employees: 15

Output:
..\data\processed\hr_intelligence.csv
